## 1. Locate source code and the Bank Marketing stream


In [ ]:
from pathlib import Path
import os, shutil, zipfile, json, sys, subprocess, time

INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
CODE = WORK_ROOT / "JCAM" / "Code"
RESULTS = CODE / "results" / "exp10_bank"
BOOTSTRAP_ERROR = None

def extract_matching_zip(target_name, destination):
    matches = []
    for path in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(path) as archive:
                if any(name.replace("\\", "/").endswith(target_name) for name in archive.namelist()):
                    matches.append(path)
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise RuntimeError(f"Expected one archive containing {target_name}, found {matches}")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0]) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        archive.extractall(destination)

try:
    source_roots = sorted({path.parent.parent for path in INPUT_ROOT.rglob("run_application.py")
                           if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if not source_roots:
        extracted_source = WORK_ROOT / "_jcam_source"
        extract_matching_zip("Code/kaggle/run_application.py", extracted_source)
        source_roots = sorted({path.parent.parent for path in extracted_source.rglob("run_application.py")
                               if path.parent.name == "kaggle" and path.parent.parent.name == "Code"})
    if len(source_roots) != 1:
        raise RuntimeError(f"Expected one Code source, found {source_roots}")
    shutil.copytree(source_roots[0], CODE, dirs_exist_ok=True)
    inputs = sorted(INPUT_ROOT.rglob("bank_stream.npz"))
    if not inputs:
        extracted_data = WORK_ROOT / "_jcam_bank_input"
        extract_matching_zip("bank_stream.npz", extracted_data)
        inputs = sorted(extracted_data.rglob("bank_stream.npz"))
    if len(inputs) != 1:
        raise RuntimeError(f"Expected one bank_stream.npz, found {inputs}")
    DATA = inputs[0]
    SOURCE_META = DATA.with_suffix(".metadata.json")
    if not SOURCE_META.is_file():
        raise FileNotFoundError(SOURCE_META)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Code:", CODE, "Data:", DATA)
except Exception as error:
    BOOTSTRAP_ERROR = repr(error)
    RESULTS.mkdir(parents=True, exist_ok=True)
    print("Bootstrap failed:", BOOTSTRAP_ERROR)


## 2. Validate the chronological NPZ


In [ ]:
import hashlib
import numpy as np

RUN_OK = BOOTSTRAP_ERROR is None
STAGES = []
ROUND_PATH = None

def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)

if RUN_OK:
    try:
        source = json.loads(SOURCE_META.read_text(encoding="utf-8"))
        if hashlib.sha256(DATA.read_bytes()).hexdigest() != source["npz_sha256"]:
            raise ValueError("NPZ SHA-256 mismatch")
        with np.load(DATA, allow_pickle=False) as data:
            x, y, g, r, w = [data[k] for k in ("features", "labels", "groups", "radii", "weights")]
            if x.ndim != 3 or y.shape != x.shape[:2] or g.shape != y.shape:
                raise ValueError("Invalid stream shapes")
            if r.shape != (len(x), 3) or w.shape != r.shape:
                raise ValueError("Invalid objective metadata shapes")
            if not np.all(np.isfinite(x)) or not np.all(np.isin(y, [-1, 1])):
                raise ValueError("Invalid features or labels")
            if not np.all((g == 0).any(axis=1) & (g == 1).any(axis=1)):
                raise ValueError("Both age groups are required in every batch")
            if not np.allclose(w.sum(axis=1), 1) or np.any(w < 0) or np.any(r < 0):
                raise ValueError("Invalid weights or radii")
        record("validate", "passed", f"T={len(x)}, n={x.shape[1]}, d={x.shape[2]}")
    except Exception as error:
        RUN_OK = False
        record("validate", "failed", repr(error))
else:
    record("validate", "skipped", BOOTSTRAP_ERROR)


## 3. Run EXP-10 in chronological order


In [ ]:
if RUN_OK:
    started = time.perf_counter()
    command = [sys.executable, str(CODE / "kaggle" / "run_application.py"),
               "--data", str(DATA), "--eta", "0.05", "--output-root", str(RESULTS)]
    outcome = subprocess.run(command, cwd=CODE, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(outcome.stdout[-5000:])
    if outcome.returncode == 0:
        checkpoints = sorted((RESULTS / "application").rglob("metadata.json"))
        if len(checkpoints) != 1:
            RUN_OK = False
            record("application", "failed", f"Expected one checkpoint, found {checkpoints}")
        else:
            ROUND_PATH = checkpoints[0].with_name("rounds.csv")
            record("application", "passed", f"seconds={time.perf_counter()-started:.2f}")
    else:
        RUN_OK = False
        record("application", "failed", f"exit={outcome.returncode}")
else:
    record("application", "skipped", "earlier gate failed")


## 4. Evaluate pre-update predictions against a causal baseline


In [ ]:
"""Audit EXP-10's pre-update predictions against a causal prevalence baseline."""

import argparse
import csv
import json
from pathlib import Path

import numpy as np


def evaluate(data_path: Path, rounds_path: Path, source_metadata: Path, output: Path) -> dict:
    with np.load(data_path, allow_pickle=False) as data:
        features = data["features"]
        labels = data["labels"]
        groups = data["groups"]
    with rounds_path.open(newline="", encoding="utf-8") as handle:
        rounds = list(csv.DictReader(handle))
    source = json.loads(source_metadata.read_text(encoding="utf-8"))
    if len(rounds) != len(features):
        raise ValueError("Round count does not match NPZ horizon")
    past_negative, past_positive = source["calibration_label_counts"]
    batch_rows = []
    total_model_loss = total_baseline_loss = total_brier = 0.0
    total_model_correct = total_baseline_correct = 0
    total_n = 0
    max_objective_difference = 0.0

    for t, row in enumerate(rounds):
        action = np.asarray(json.loads(row["action"]), dtype=float)
        if action.shape != (features.shape[2],):
            raise ValueError(f"Action dimension mismatch at batch {t}")
        x = np.asarray(features[t], dtype=float)
        y = np.asarray(labels[t], dtype=float)
        group = np.asarray(groups[t], dtype=int)
        scores = x @ action
        margins = y * scores
        model_loss = np.logaddexp(0, -margins)
        probability = 1 / (1 + np.exp(-np.clip(scores, -700, 700)))
        baseline_probability = np.clip(past_positive / (past_positive + past_negative), 1e-12, 1 - 1e-12)
        baseline_logit = float(np.log(baseline_probability / (1 - baseline_probability)))
        baseline_loss = np.logaddexp(0, -y * baseline_logit)
        observed_midpoint = json.loads(row["midpoints"])[0]
        max_objective_difference = max(max_objective_difference, abs(float(observed_midpoint) - float(model_loss.mean())))
        if max_objective_difference > 1e-6:
            raise ValueError("Logged prequential logistic objective does not match recomputation")
        n = len(y)
        gap = abs(float(scores[group == 0].mean() - scores[group == 1].mean()))
        batch_rows.append({
            "batch": t + 1, "n": n, "positives": int(np.sum(y > 0)),
            "group0": int(np.sum(group == 0)), "group1": int(np.sum(group == 1)),
            "model_logloss": float(model_loss.mean()),
            "causal_prevalence_baseline_logloss": float(baseline_loss.mean()),
            "model_accuracy": float(np.mean((scores >= 0) == (y > 0))),
            "baseline_accuracy": float(np.mean((baseline_logit >= 0) == (y > 0))),
            "model_brier": float(np.mean((probability - (y > 0)) ** 2)),
            "score_group_gap": gap,
            "baseline_pre_batch_positive_rate": baseline_probability,
        })
        total_n += n
        total_model_loss += float(model_loss.sum())
        total_baseline_loss += float(baseline_loss.sum())
        total_brier += float(np.sum((probability - (y > 0)) ** 2))
        total_model_correct += int(np.sum((scores >= 0) == (y > 0)))
        total_baseline_correct += int(np.sum((baseline_logit >= 0) == (y > 0)))
        past_positive += int(np.sum(y > 0))
        past_negative += int(np.sum(y < 0))

    output.mkdir(parents=True, exist_ok=True)
    with (output / "per_batch.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(batch_rows[0]))
        writer.writeheader()
        writer.writerows(batch_rows)
    summary = {
        "status": "success", "interpretation": "exploratory, prequential empirical metrics",
        "n": total_n, "batches": len(batch_rows),
        "model_logloss": total_model_loss / total_n,
        "causal_prevalence_baseline_logloss": total_baseline_loss / total_n,
        "model_accuracy": total_model_correct / total_n,
        "baseline_accuracy": total_baseline_correct / total_n,
        "model_brier": total_brier / total_n,
        "mean_batch_score_group_gap": float(np.mean([r["score_group_gap"] for r in batch_rows])),
        "max_logged_logloss_difference": max_objective_difference,
        "population_interval_coverage_verified": False,
    }
    (output / "evaluation.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary

if RUN_OK:
    try:
        SUMMARY = evaluate(DATA, ROUND_PATH, SOURCE_META, RESULTS / "evaluation")
        print(json.dumps(SUMMARY, indent=2))
        record("evaluation", "passed", "prequential metrics saved")
    except Exception as error:
        RUN_OK = False
        record("evaluation", "failed", repr(error))
else:
    record("evaluation", "skipped", "earlier gate failed")


## 5. Plot diagnostics and zip all EXP-10 outputs


In [ ]:
import csv, zipfile

if RUN_OK:
    try:
        import matplotlib.pyplot as plt
        with (RESULTS / "evaluation" / "per_batch.csv").open(newline="", encoding="utf-8") as handle:
            per_batch = list(csv.DictReader(handle))
        steps = [int(row["batch"]) for row in per_batch]
        fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
        ax[0].plot(steps, [float(row["model_logloss"]) for row in per_batch], label="Online midpoint")
        ax[0].plot(steps, [float(row["causal_prevalence_baseline_logloss"]) for row in per_batch], label="Past-prevalence baseline")
        ax[0].set_ylabel("Pre-update log loss")
        ax[0].legend()
        ax[1].plot(steps, [float(row["score_group_gap"]) for row in per_batch])
        ax[1].set_ylabel("Absolute age-group mean score gap")
        ax[1].set_xlabel("Chronological batch")
        fig.tight_layout()
        fig.savefig(RESULTS / "exp10_diagnostics.png", dpi=160)
        plt.close(fig)
        record("plot", "passed", "exp10_diagnostics.png")
    except Exception as error:
        RUN_OK = False
        record("plot", "failed", repr(error))
else:
    record("plot", "skipped", "earlier gate failed")

(RESULTS / "notebook_stages.json").write_text(json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
if BOOTSTRAP_ERROR is None:
    shutil.copy2(SOURCE_META, RESULTS / "bank_stream.metadata.json")
FINAL_ZIP = WORK_ROOT / "result_jcam_exp10.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=(Path("exp10_bank") / path.relative_to(RESULTS)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
